In [ ]:
import requests
import math
import numpy as np
import pandas as pd
from itertools import product
from typing import Dict, Tuple, List, Union
import matplotlib.pyplot as plt

In [ ]:
symbols = [
    "NABIL", "GBIME", "EBL", "NIMB", "SCB", "LSL", "SBL", "KBL", "PRVU", "NICA",
    "PCBL", "NMB", "HBL", "ADBL", "SBI", "SANIMA", "NBL", "CZBIL", "SHL", "CHCL",
    "UPPER", "CIT", "HIDCL", "NIFRA", "NLIC", "LICN", "HLI", "NLICL", "UNL", "SARBTM",
    "HDL", "SHIVM", "BNL", "SKBBL", "CBBL", "RBCL", "NRIC", "NTC", "HRL", "BBC"
]

In [ ]:
for symbol in symbols:
    url = f"https://sharehubnepal.com/data/api/v1/candle-chart/history?symbol={symbol}&resolution=1D&countback=0&isAdjust=true"
    response = requests.get(url)
    
    if response.status_code == 200:
        data = response.json()
        df = pd.json_normalize(data['data'])
        df = df.sort_values('time').reset_index(drop=True)
        df = df.iloc[15:].reset_index(drop=True) 
        df['close'] = pd.to_numeric(df['close'], errors='coerce')
        
        globals()[symbol] = df
        print(f"{symbol} dataframe created with {len(df)} rows")
    else:
        print(f"Failed to fetch data for {symbol}, status code: {response.status_code}")

In [ ]:
datasets = {
    "NABIL": NABIL, "GBIME": GBIME, "EBL": EBL, "NIMB": NIMB, "SCB": SCB, "LSL": LSL, "SBL": SBL, "KBL": KBL, "PRVU": PRVU, "NICA": NICA,
    "PCBL": PCBL, "NMB": NMB, "HBL": HBL, "ADBL": ADBL, "SBI": SBI, "SANIMA": SANIMA, "NBL": NBL, "CZBIL": CZBIL, "SHL": SHL, "CHCL": CHCL,
    "UPPER": UPPER, "CIT": CIT, "HIDCL": HIDCL, "NIFRA": NIFRA, "NLIC": NLIC, "LICN": LICN, "HLI": HLI, "NLICL": NLICL, "UNL": UNL, "SARBTM": SARBTM,
    "HDL": HDL, "SHIVM": SHIVM
}

In [ ]:
def _ensure_close_df(
    prices: Union[pd.DataFrame, Dict[str, pd.DataFrame]]
) -> pd.DataFrame:
    """
    Accepts either:
      - a DataFrame with close prices as columns (index: dates),
      - or a dict of DataFrames where each DataFrame has a 'Close' column.
    Returns a DataFrame of closes with columns = symbols.
    """
    if isinstance(prices, pd.DataFrame):
        return prices.copy()
    elif isinstance(prices, dict):
        close_dict = {}
        for symbol, df in prices.items():
            if isinstance(df, pd.Series):
                close_dict[symbol] = df.rename(symbol)
            elif "close" in df.columns:
                close_dict[symbol] = df["close"].rename(symbol)
            else:
                raise ValueError(f"DataFrame for {symbol} must contain 'Close' column or be a Series.")
        close_df = pd.concat(close_dict.values(), axis=1)
        close_df.columns = list(close_dict.keys())
        return close_df
    else:
        raise ValueError("prices must be a DataFrame or a dict of DataFrames")

In [ ]:
def ema(
    series: pd.Series,
    window: int
) -> pd.Series:
    return series.ewm(span=window, adjust=False, min_periods=1).mean()

In [ ]:
def sma_crossover_returns_for_symbol(
    close: pd.Series,
    n1: int,
    n2: int,
    min_periods: int = None
) -> pd.Series:
    """
    Simulate simple SMA crossover trades for one symbol.
    - n1: short SMA window
    - n2: long SMA window
    Returns a Series of same index where:
      - on the selling day (exit) the trade return (simple return) is placed,
      - all other days are 0.
    Rules:
      - Enter long when SMA_short > SMA_long (signal = 1)
      - Exit when SMA_short <= SMA_long (signal becomes 0)
      - Entry price is the close at the day the signal flipped to 1 (we treat entry at that day's close),
        Exit price is close at the day signal flips to 0 (we place return on the exit day's row).
      - If a position is open at the end, we close at the last available close and record that return on the last index row.
    """
    if min_periods is None:
        min_periods = 1

    short = close.ewm(span=n1, adjust=False, min_periods=min_periods).mean()
    long  = close.ewm(span=n2, adjust=False, min_periods=min_periods).mean()

    # True when short > long 
    signal = (short > long).astype(int)

    returns = pd.Series(0.0, index=close.index)

    in_position = False
    entry_price = None
    entry_idx = None

    # Iterate index-wise
    prev_sig = 0
    for idx in close.index:
        sig = int(signal.loc[idx]) if not pd.isna(signal.loc[idx]) else 0

        # Entry: prev_sig==0 and sig==1
        if (prev_sig == 0) and (sig == 1):
            in_position = True
            entry_price = close.loc[idx]  # enter at close of this day
            entry_idx = idx

        # Exit: prev_sig==1 and sig==0 
        if (prev_sig == 1) and (sig == 0) and in_position:
            exit_price = close.loc[idx]
            # simple return
            r = (exit_price - entry_price) / entry_price if entry_price != 0 else 0.0
            returns.loc[idx] = r
            in_position = False
            entry_price = None
            entry_idx = None

        prev_sig = sig

    # If still in position at the end, close at last available close
    if in_position and entry_price is not None:
        last_idx = close.index[-1]
        exit_price = close.loc[last_idx]
        r = (exit_price - entry_price) / entry_price if entry_price != 0 else 0.0
        returns.loc[last_idx] = returns.loc[last_idx] + r 
        in_position = False

    return returns

In [ ]:
def compute_returns_dataframe(
    closes: Union[pd.DataFrame, Dict[str, pd.DataFrame]],
    n1: int,
    n2: int,
) -> pd.DataFrame:
    """
    Create a DataFrame of returns where each column corresponds to a symbol.
    Returns are placed on exit days; other rows are zero.
    """
    close_df = _ensure_close_df(closes)
    returns_df = pd.DataFrame(0.0, index=close_df.index, columns=close_df.columns)

    for col in close_df.columns:
        returns_df[col] = sma_crossover_returns_for_symbol(close_df[col].dropna(), n1=n1, n2=n2)
        returns_df[col] = returns_df[col].reindex(close_df.index, fill_value=0.0)

    return returns_df

In [ ]:
def custom_sharpe_from_returns_series(
    returns: pd.Series, 
    N: int = 252
) -> float:
    """
    Calculate Sharpe using:
      - average daily return = mean of the returns series (as the user requested)
      - volatility = population standard deviation (ddof=0)
      - Sharpe = (mean / sigma) * N  
    If sigma == 0 -> returns np.nan
    """
    mean_daily = returns.mean()
    sigma = returns.std(ddof=0)  
    if sigma == 0 or np.isclose(sigma, 0.0):
        return np.nan
    return (mean_daily / sigma) * math.sqrt(N)

In [ ]:
def custom_sharpe_for_returns_df(
    returns_df: pd.DataFrame, 
    N: int = 252
) -> pd.Series:
    """
    Compute the custom sharpe for each column (symbol) in returns_df.
    Returns a Series indexed by symbol.
    """
    return returns_df.apply(lambda col: custom_sharpe_from_returns_series(col, N=N))

In [ ]:
def optimize_sma_sharpe(
    closes: Union[pd.DataFrame, Dict[str, pd.DataFrame]],
    n1_values: List[int],
    n2_values: List[int],
    N: int = 252,
    verbose: bool = False
) -> Tuple[Tuple[int, int], pd.DataFrame]:
    """
    Grid-search n1 (short) and n2 (long) over provided lists to maximize aggregate Sharpe.
    Returns:
    Best_pair: (best_n1, best_n2)
    Results_df: DataFrame with columns ['n1','n2','avg_sharpe','sum_sharpe','sharpe_by_symbol' (object)]
    """
    close_df = _ensure_close_df(closes)
    results = []

    for n1, n2 in product(n1_values, n2_values):
        if n1 >= n2:
            continue  

        returns_df = compute_returns_dataframe(close_df, n1=n1, n2=n2)
        sharpe_series = custom_sharpe_for_returns_df(returns_df, N=N)

        avg_sharpe = sharpe_series.mean(skipna=True)
        sum_sharpe = sharpe_series.sum(skipna=True)

        results.append({
            "n1": n1,
            "n2": n2,
            "avg_sharpe": avg_sharpe,
            "sum_sharpe": sum_sharpe,
            "sharpe_by_symbol": sharpe_series
        })

        if verbose:
            print(f"n1={n1}, n2={n2}, avg_sharpe={avg_sharpe:.6f}, sum_sharpe={sum_sharpe:.6f}")

    results_df = pd.DataFrame(results)
    if results_df.empty:
        raise ValueError("No valid (n1,n2) pairs tested. Ensure n1_values and n2_values contain n1 < n2 combos.")

    idx_best = results_df["avg_sharpe"].idxmax()
    best_row = results_df.loc[idx_best]
    best_pair = (int(best_row["n1"]), int(best_row["n2"]))

    return best_pair, results_df

In [ ]:
def evaluate_ma_strategy(
    df: pd.DataFrame,
    n1: int, 
    n2: int
) -> pd.Series:
    
    # Compute EMAs
    df = df.copy()
    df['EMA_fast'] = df['close'].ewm(span=n1, adjust=False).mean()
    df['EMA_slow'] = df['close'].ewm(span=n2, adjust=False).mean()
    
    # Trade Signals
    df['signal'] = 0
    df.loc[(df['EMA_fast'] > df['EMA_slow']) & (df['EMA_fast'].shift(1) <= df['EMA_slow'].shift(1)), 'signal'] = 1  # BUY
    df.loc[(df['EMA_fast'] < df['EMA_slow']) & (df['EMA_fast'].shift(1) >= df['EMA_slow'].shift(1)), 'signal'] = -1 # SELL
    
    # Track trades 
    trades = []
    in_pos = False
    entry_price = None

    # Track trades & drawdowns 
    trades = []
    drawdowns = []        

    in_pos = False
    entry_price = None
    peak = None
    trough = None

    for idx, row in df.iterrows():

        # Enter trade
        if row['signal'] == 1 and not in_pos:
            entry_price = row['close']
            peak = entry_price
            trough = entry_price
            in_pos = True

        # While in trade, update peak/trough
        elif in_pos and row['signal'] == 0:
            price = row['close']

            # Update peak if price exceeds previous peak
            if price > peak:
                peak = price
                trough = peak  # reset trough to new peak
            # Update trough only if price falls below current trough AFTER peak
            elif price < trough:
                trough = price

        # Exit trade
        elif row['signal'] == -1 and in_pos:
            exit_price = row['close']

            # Calculate trade % return
            ret_pct = (exit_price - entry_price) / entry_price * 100.0
            trades.append(ret_pct)

            # Calculate max drawdown inside this trade
            mdd = (peak - trough) / peak * 100.0
            drawdowns.append(mdd)

            in_pos = False
            
    max_drawdown_over_trades = max(drawdowns) if drawdowns else 0

    # Performance 
    avg_trade_pct = np.sum(trades) if trades else 0
    num_trades    = len(trades)

    # Compute Sharpe from daily returns 
    df['daily_ret'] = 0.0
    trade_idx = 0
    for i, row in df.iterrows():
        # Assign trade return only on sell days
        if trade_idx < len(trades) and row['signal'] == -1:
            df.at[i, 'daily_ret'] = trades[trade_idx] / 100.0  # convert to decimal
            trade_idx += 1

    avg_daily = df['daily_ret'].mean()
    vol_daily = df['daily_ret'].std(ddof=1)
    sharpe = (avg_daily / vol_daily) * math.sqrt(252) if vol_daily != 0 else 0

    # Buy & Hold
    bh_pct = (df['close'].iloc[-1] - df['close'].iloc[0]) / df['close'].iloc[0] * 100

    out = {
        "Total Profit" : avg_trade_pct,
        "Num Trades"               : num_trades,
        "Annualized Sharpe"        : sharpe,
        "Buy & Hold Return (%)"    : bh_pct,
        "Max Drawdown (%)" : max_drawdown_over_trades
    }
    return pd.Series(out)

In [ ]:
def evaluate_on_test(
    test_sets: dict, 
    n1: int, 
    n2: int, 
    N: int = 252
):
    """
    Evaluate SMA crossover vs Buy & Hold on testing data.
    Returns average metrics across all symbols:
    Annual Profit %
    Annualized Sharpe Ratio
    Max Drawdown %
    """
    crossover_profits = []
    crossover_sharpes = []
    crossover_drawdowns = []
    crossover_trades = []

    bh_profits = []
    bh_sharpes = []
    bh_drawdowns = []

    for symbol, df in test_sets.items():
        if df is None or df.empty or "close" not in df.columns:
            continue

        # EMA Crossover 
        metrics = evaluate_ma_strategy(df, n1, n2)
        crossover_profits.append(metrics["Total Profit"])
        crossover_sharpes.append(metrics["Annualized Sharpe"])
        crossover_drawdowns.append(metrics["Max Drawdown (%)"])
        crossover_trades.append(metrics["Num Trades"])

        # Buy & Hold
        close = df['close']
        days = len(close)
        bh_return_pct = (close.iloc[-1] - close.iloc[0]) / close.iloc[0] * 100
        bh_annualized = bh_return_pct * (252 / days)  # annualize
        bh_profits.append(bh_annualized)

        bh_profits.append(bh_return_pct)

        # Daily returns for Buy & Hold
        daily_ret = close.pct_change().fillna(0)
        mean_daily = daily_ret.mean()
        sigma = daily_ret.std(ddof=1)
        bh_sharpe = (mean_daily / sigma) * math.sqrt(N) if sigma != 0 else 0
        bh_sharpes.append(bh_sharpe)

        # Max drawdown for Buy & Hold
        cum_max = close.cummax()
        drawdown = (cum_max - close) / cum_max * 100
        bh_drawdowns.append(drawdown.max())

    result = {
        "Crossover Profit % (avg)" : np.mean(crossover_profits),
        "Crossover Sharpe (avg)"   : np.mean(crossover_sharpes),
        "Crossover Max Drawdown %" : np.mean(crossover_drawdowns),
        "Number of Trades (avg)": np.mean(crossover_trades),
        "Buy & Hold Profit % (avg)": np.mean(bh_profits),
        "Buy & Hold Sharpe (avg)"  : np.mean(bh_sharpes),
        "Buy & Hold Max Drawdown %" : np.mean(bh_drawdowns)
    }

    return pd.Series(result)

In [ ]:
n1_vals = list(range(5, 20))
n2_vals = list(range(21, 40))
best_pair, train_res_df = optimize_sma_sharpe(datasets, n1_vals, n2_vals, verbose=True)

In [ ]:
oos_datasets = {
    "BNL": BNL, "SKBBL": SKBBL, "CBBL": CBBL, "RBCL": RBCL, "NRIC": NRIC, "NTC": NTC, "HRL": HRL, "BBC": BBC
}

In [ ]:
res = evaluate_ma_strategy(JBLB, best_pair[0], best_pair[1])

In [ ]:
BROKER_TIERS = [
    (0, 50_000, 0.0036, 10.0),
    (50_001, 500_000, 0.0033, None),
    (500_001, 2_000_000, 0.0031, None),
    (2_000_001, 10_000_000, 0.0027, None),
    (10_000_001, float("inf"), 0.0024, None)
]
SEBON_RATE = 0.00015     # 0.015%
DP_CHARGE = 25.0         # Rs 25 on sell
SHORT_TERM_TAX = 0.075
LONG_TERM_TAX  = 0.05
TRADING_DAYS_YEAR = 252

In [ ]:
def broker_commission_for_amount(
    amount: float
) -> float:
    for lo, hi, pct, flat in BROKER_TIERS:
        if lo <= amount <= hi:
            if flat is not None:
                pct_amt = amount * pct
                return max(flat, pct_amt)
            return amount * pct
    return amount * BROKER_TIERS[-1][2]

In [ ]:
def compute_trade_net_return(
    entry_price: float, 
    exit_price: float, 
    capital: float, 
    holding_days: int
) -> float:
    """
    Returns net return as a decimal relative to invested capital (entry_price * qty).
    Simulates buying floor(capital/entry_price) shares and selling them,
    subtracting broker commissions (buy+sell), SEBON (buy+sell), DP on sell,
    then capital gains tax on net profit (if profit > 0).
    """
    if entry_price <= 0:
        return 0.0
    qty = math.floor(capital / entry_price)
    if qty <= 0:
        return 0.0
    invest_amount = entry_price * qty
    sell_amount   = exit_price * qty

    buy_broker = broker_commission_for_amount(invest_amount)
    buy_sebon  = invest_amount * SEBON_RATE

    sell_broker = broker_commission_for_amount(sell_amount)
    sell_sebon  = sell_amount * SEBON_RATE

    cash_out = sell_amount - sell_broker - sell_sebon - DP_CHARGE
    cash_in  = invest_amount + buy_broker + buy_sebon

    profit_before_tax = cash_out - cash_in

    tax = 0.0
    if profit_before_tax > 0:
        tax_rate = SHORT_TERM_TAX if holding_days < TRADING_DAYS_YEAR else LONG_TERM_TAX
        tax = profit_before_tax * tax_rate

    net_profit = profit_before_tax - tax
    net_return_decimal = net_profit / invest_amount if invest_amount != 0 else 0.0
    return net_return_decimal

In [ ]:
def extract_trade_points_from_signals(df):
    """
    Expect df['signal'] coded as:
    1 on buy (entry) day, -1 on sell (exit) day.
    Returns aligned lists: entries, exits (exit may be None -> close at end).
    """
    entries = []
    exits = []
    in_pos = False
    current_entry = None
    for idx, row in df.iterrows():
        sig = int(row.get('signal', 0))
        if sig == 1 and not in_pos:
            in_pos = True
            current_entry = idx
        if sig == -1 and in_pos:
            entries.append(current_entry)
            exits.append(idx)
            in_pos = False
            current_entry = None
    # If still in position at end
    if in_pos and current_entry is not None:
        entries.append(current_entry)
        exits.append(None)
    return entries, exits

In [ ]:
def _trade_based_sharpe_from_series(trade_returns_decimal: pd.Series, N:int=252):
    """
    Compute Sharpe from a series where most days are 0 and SELL days
    contain trade returns as decimals. Use population std (ddof=0).
    Returns np.nan if sigma == 0 or insufficient data.
    """
    if trade_returns_decimal is None or len(trade_returns_decimal) < 2:
        return np.nan
    mean_daily = trade_returns_decimal.mean()
    sigma = trade_returns_decimal.std(ddof=0) 
    if sigma == 0 or np.isclose(sigma, 0.0):
        return np.nan
    return (mean_daily / sigma) * math.sqrt(N)

In [ ]:
def evaluate_ma_strategy_real(
    df: pd.DataFrame, 
    n1: int, 
    n2: int, 
    capital=50_000
):
    """
    For a single df (must contain 'close'), produce BOTH ideal and real metrics.
    Ideal: trades use (exit-entry)/entry placed on exit dates.
    Real: trades use compute_trade_net_return (which includes fees/tax) placed on exit dates.
    Buy&Hold treated as one trade (buy at iloc[14], sell at iloc[-1]).
    Returns: {"ideal": pd.Series(...), "real": pd.Series(...)}
    Keys in Series:
    "Total Profit", "Num Trades", "Annualized Sharpe", "Buy & Hold Return (%)", "Buy & Hold Sharpe", "Max Drawdown (%)"
    """
    dfc = df.copy()
    dfc['EMA_fast'] = dfc['close'].ewm(span=n1, adjust=False).mean()
    dfc['EMA_slow'] = dfc['close'].ewm(span=n2, adjust=False).mean()

    # Signals
    dfc['signal'] = 0
    dfc.loc[(dfc['EMA_fast'] > dfc['EMA_slow']) & (dfc['EMA_fast'].shift(1) <= dfc['EMA_slow'].shift(1)), 'signal'] = 1
    dfc.loc[(dfc['EMA_fast'] < dfc['EMA_slow']) & (dfc['EMA_fast'].shift(1) >= dfc['EMA_slow'].shift(1)), 'signal'] = -1

    entries, exits = extract_trade_points_from_signals(dfc)

    # Ideal Trade Returns
    ideal_trade_returns = pd.Series(0.0, index=dfc.index)
    ideal_trades_list_pct = []
    for e, x in zip(entries, exits):
        exit_idx = dfc.index[-1] if x is None else x
        p_entry = float(dfc.loc[e, 'close'])
        p_exit  = float(dfc.loc[exit_idx, 'close'])
        r = (p_exit - p_entry) / p_entry if p_entry != 0 else 0.0
        ideal_trade_returns.loc[exit_idx] = r
        ideal_trades_list_pct.append(r * 100.0)

    # Real Trade Returns
    real_trade_returns = pd.Series(0.0, index=dfc.index)
    for e, x in zip(entries, exits):
        exit_idx = dfc.index[-1] if x is None else x
        p_entry = float(dfc.loc[e, 'close'])
        p_exit  = float(dfc.loc[exit_idx, 'close'])
        holding_days = dfc.index.get_loc(exit_idx) - dfc.index.get_loc(e)
        net_decimal = compute_trade_net_return(p_entry, p_exit, capital, holding_days)
        real_trade_returns.loc[exit_idx] = net_decimal

    # Summary stats for Crossover
    
    # Ideal
    num_trades = len(ideal_trades_list_pct)
    total_profit_pct_ideal = np.sum(ideal_trades_list_pct) if ideal_trades_list_pct else 0.0
    sharpe_ideal = _trade_based_sharpe_from_series(ideal_trade_returns, N=TRADING_DAYS_YEAR)

    # Real
    total_profit_pct_real = real_trade_returns.sum() * 100.0  # sum of decimals -> percent
    num_trades_real = (real_trade_returns != 0).sum()
    sharpe_real = _trade_based_sharpe_from_series(real_trade_returns, N=TRADING_DAYS_YEAR)

    # Buy & hold
    if len(dfc) <= 14:
        bh_pct_ideal = 0.0
        bh_pct_real  = 0.0
    else:
        buy_idx = 14
        sell_idx = dfc.index[-1]
        p_buy = float(dfc.loc[buy_idx, 'close'])
        p_sell = float(dfc.loc[sell_idx, 'close'])
        # Ideal BH decimal
        bh_decimal_ideal = (p_sell - p_buy) / p_buy if p_buy != 0 else 0.0
        bh_pct_ideal = bh_decimal_ideal * 100.0
        # Real BH using capital sizing
        holding_days_bh = dfc.index.get_loc(sell_idx) - dfc.index.get_loc(buy_idx)
        bh_decimal_real = compute_trade_net_return(p_buy, p_sell, capital, holding_days_bh)
        bh_pct_real = bh_decimal_real * 100.0

    # Place BH return on last day
    ideal_bh_series = pd.Series(0.0, index=dfc.index); ideal_bh_series.iloc[-1] = bh_decimal_ideal if len(dfc) > 14 else 0.0
    real_bh_series  = pd.Series(0.0, index=dfc.index); real_bh_series.iloc[-1] = bh_decimal_real if len(dfc) > 14 else 0.0

    bh_sharpe_ideal = _trade_based_sharpe_from_series(ideal_bh_series, N=TRADING_DAYS_YEAR)
    bh_sharpe_real  = _trade_based_sharpe_from_series(real_bh_series, N=TRADING_DAYS_YEAR)

    # Max Drawdown
    drawdowns = []
    in_pos = False
    entry_price = None
    peak = trough = None
    for idx, row in dfc.iterrows():
        if row['signal'] == 1 and not in_pos:
            entry_price = row['close']; peak = entry_price; trough = entry_price; in_pos = True
        elif in_pos and row['signal'] == 0:
            price = row['close']
            if price > peak:
                peak = price; trough = peak
            elif price < trough:
                trough = price
        elif row['signal'] == -1 and in_pos:
            mdd = (peak - trough) / peak * 100.0 if peak and trough else 0.0
            drawdowns.append(mdd)
            in_pos = False
    max_drawdown = max(drawdowns) if drawdowns else 0.0

    ideal_out = {
        "Total Profit"            : total_profit_pct_ideal,
        "Num Trades"              : num_trades,
        "Annualized Sharpe"       : sharpe_ideal,
        "Buy & Hold Return (%)"   : bh_pct_ideal,
        "Buy & Hold Sharpe"       : bh_sharpe_ideal,
        "Max Drawdown (%)"        : max_drawdown
    }

    real_out = {
        "Total Profit"            : total_profit_pct_real,
        "Num Trades"              : int(num_trades_real),
        "Annualized Sharpe"       : sharpe_real,
        "Buy & Hold Return (%)"   : bh_pct_real,
        "Buy & Hold Sharpe"       : bh_sharpe_real,
        "Max Drawdown (%)"        : max_drawdown
    }

    return {"ideal": pd.Series(ideal_out), "real": pd.Series(real_out)}

In [ ]:
def evaluate_on_test_with_costs(
    test_sets: dict, 
    n1: int, 
    n2: int, 
    capital_values=[50_000, 10_000_000]
):
    """
    Aggregate across symbols. Returns pd.Series with keys:
    Crossover Profit % (avg) - ideal
    Crossover Sharpe (avg) - ideal
    And the same keys for each capital scenario with prefix 'real_cap_<cap>'
    """
    ideal_acc = {"profit": [], "sharpe": [], "mdd": [], "trades": [], "bh_profit": [], "bh_sharpe": []}
    real_acc = {cap: {"profit": [], "sharpe": [], "mdd": [], "trades": [], "bh_profit": [], "bh_sharpe": []} for cap in capital_values}

    for symbol, df in test_sets.items():
        if df is None or df.empty or "close" not in df.columns:
            continue
        result_ideal_real = evaluate_ma_strategy_real(df, n1, n2, capital=capital_values[0])
        ideal = result_ideal_real['ideal']
        ideal_acc["profit"].append(ideal["Total Profit"])
        ideal_acc["sharpe"].append(ideal["Annualized Sharpe"])
        ideal_acc["mdd"].append(ideal["Max Drawdown (%)"])
        ideal_acc["trades"].append(ideal["Num Trades"])
        ideal_acc["bh_profit"].append(ideal["Buy & Hold Return (%)"])
        ideal_acc["bh_sharpe"].append(ideal["Buy & Hold Sharpe"])

        for cap in capital_values:
            resr = evaluate_ma_strategy_real(df, n1, n2, capital=cap)['real']
            d = real_acc[cap]
            d["profit"].append(resr["Total Profit"])
            d["sharpe"].append(resr["Annualized Sharpe"])
            d["mdd"].append(resr["Max Drawdown (%)"])
            d["trades"].append(resr["Num Trades"])
            d["bh_profit"].append(resr["Buy & Hold Return (%)"])
            d["bh_sharpe"].append(resr["Buy & Hold Sharpe"])

    out = {}

    out["Crossover Profit % (avg) - ideal"] = np.nanmean(ideal_acc["profit"]) if ideal_acc["profit"] else 0.0
    out["Crossover Sharpe (avg) - ideal"] = np.nanmean(ideal_acc["sharpe"]) if ideal_acc["sharpe"] else np.nan
    out["Crossover Max Drawdown % - ideal"] = np.nanmean(ideal_acc["mdd"]) if ideal_acc["mdd"] else 0.0
    out["Number of Trades (avg) - ideal"] = np.nanmean(ideal_acc["trades"]) if ideal_acc["trades"] else 0.0
    out["Buy & Hold Profit % (avg) - ideal"] = np.nanmean(ideal_acc["bh_profit"]) if ideal_acc["bh_profit"] else 0.0
    out["Buy & Hold Sharpe (avg) - ideal"] = np.nanmean(ideal_acc["bh_sharpe"]) if ideal_acc["bh_sharpe"] else np.nan
    out["Buy & Hold Max Drawdown % - ideal"] = np.nanmean(ideal_acc["mdd"]) if ideal_acc["mdd"] else 0.0

    for cap in capital_values:
        d = real_acc[cap]
        prefix = f"real_cap_{int(cap)}"
        out[f"Crossover Profit % (avg) - {prefix}"] = np.nanmean(d["profit"]) if d["profit"] else 0.0
        out[f"Crossover Sharpe (avg) - {prefix}"] = np.nanmean(d["sharpe"]) if d["sharpe"] else np.nan
        out[f"Crossover Max Drawdown % - {prefix}"] = np.nanmean(d["mdd"]) if d["mdd"] else 0.0
        out[f"Number of Trades (avg) - {prefix}"] = np.nanmean(d["trades"]) if d["trades"] else 0.0
        out[f"Buy & Hold Profit % (avg) - {prefix}"] = np.nanmean(d["bh_profit"]) if d["bh_profit"] else 0.0
        out[f"Buy & Hold Sharpe (avg) - {prefix}"] = np.nanmean(d["bh_sharpe"]) if d["bh_sharpe"] else np.nan
        out[f"Buy & Hold Max Drawdown % - {prefix}"] = np.nanmean(d["mdd"]) if d["mdd"] else 0.0

    return pd.Series(out)

In [ ]:
oos_metrics_both_fixed = evaluate_on_test_with_costs(oos_datasets, best_pair[0], best_pair[1], capital_values=[50_000, 10_000_000])